In [2]:
# importation et chargement
import pandas as pd

df = pd.read_csv("../data/regularite-mensuelle-transilien(data brute).csv", sep=";")

df.head()

,Date,Service,Ligne,Nom de la ligne,Taux de ponctualité,Nombre de voyageurs à l'heure pour un voyageur en retard
0,2026-10,RER,E,RER E,89.564,8.582
1,2026-10,Transilien,L,Paris Saint-Lazare Sud,92.096,11.652
2,2026-10,RER,B,RER B,83.323,4.996
3,2026-10,RER,D,RER D,78.380,3.625
4,2026-10,RER,C,RER C,85.595,5.942
5,2026-10,Transilien,J,Paris Saint-Lazare Nord,91.274,10.459
6,2026-10,Transilien,K,Paris Nord Crépy,92.935,13.154


In [3]:
# inspection des datas
print("Dimensions :", df.shape)

print("\nColonnes :")
print(df.columns)

print("\nInformations :")
df.info()

print("\nValeurs manquantes :")
print(df.isnull().sum())

print("\nNombre de doublons :")
print(df.duplicated().sum())

# le dataset contient donc 2100 lignes avec des données qui sont du type string et float et 0 doublon. Neanmois, il manque une donnee a la ponctualité.

Dimensions : (2100, 6)

Colonnes :
Index(['Date', 'Service', 'Ligne', 'Nom de la ligne', 'Taux de ponctualité',
       'Nombre de voyageurs à l'heure pour un voyageur en retard'],
      dtype='str')

Informations :
<class 'pandas.DataFrame'>
RangeIndex: 2100 entries, 0 to 2099
Data columns (total 6 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0   Date                                                      2100 non-null   str    
 1   Service                                                   2100 non-null   str    
 2   Ligne                                                     2100 non-null   str    
 3   Nom de la ligne                                           2100 non-null   str    
 4   Taux de ponctualité                                       2099 non-null   float64
 5   Nombre de voyageurs à l'heure pour un voyageur en retard  2100 non-null   fl

In [4]:
# uniformisation des noms
import unicodedata

df.columns = [
    unicodedata.normalize("NFKD", col)
    .encode("ascii", "ignore")
    .decode("ascii")
    .strip()
    .lower()
    .replace(" ", "_")
    for col in df.columns
]

print(df.columns)

Index(['date', 'service', 'ligne', 'nom_de_la_ligne', 'taux_de_ponctualite',
       'nombre_de_voyageurs_a_l'heure_pour_un_voyageur_en_retard'],
      dtype='str')


In [5]:
# filtrage transilien des RERs
df = df[df["service"] == "Transilien"].copy()

print(df["ligne"].unique())

<StringArray>
['L', 'J', 'K', 'U', 'H', 'N', 'P', 'R']
Length: 8, dtype: str


In [6]:
#filtrage des colonnes utilies
df = df[[
    "date",
    "ligne",
    "taux_de_ponctualite"
]]

df.head()

,date,ligne,taux_de_ponctualite
1,2026-10,L,92.096
5,2026-10,J,91.274
6,2026-10,K,92.935
7,2026-10,U,94.150
8,2026-10,H,92.869


In [9]:
# nettoyage des datas
df["date"] = pd.to_datetime(
    df["date"], errors="coerce"
)

df["taux_de_ponctualite"] = pd.to_numeric(
    df["taux_de_ponctualite"]
    .astype(str)
    .str.replace(",", ".")
    .str.replace("%", "")
    .str.strip(),
    errors="coerce"
)

print(df.isnull().sum())

df = df[
    df["taux_de_ponctualite"].between(0, 100)
]

date                   0
ligne                  0
taux_de_ponctualite    0
dtype: int64


In [10]:
# creation de nouvelles variables a partir des datas existantes
df["annee"] = df["date"].dt.year
df["mois"] = df["date"].dt.month

df["trimestre"] = df["date"].dt.quarter

df.head()

,date,ligne,taux_de_ponctualite,annee,mois,trimestre
1,2026-10-01,L,92.096,2026,10,4
5,2026-10-01,J,91.274,2026,10,4
6,2026-10-01,K,92.935,2026,10,4
7,2026-10-01,U,94.150,2026,10,4
8,2026-10-01,H,92.869,2026,10,4


In [12]:
# tri des datas
df = df.sort_values(
    by=["date", "ligne"]
)

df = df.reset_index(drop=True)

In [13]:
# verification du resultat
print("Dimensions finales :", df.shape)

print("\nValeurs manquantes :")
print(df.isnull().sum())

print("\nDoublons :")
print(df.duplicated().sum())

print("\nDoublons par date et ligne :")
print(df.duplicated(
    subset=["date", "ligne"]
).sum())

print("\nPériode couverte :")
print(df["date"].min())
print(df["date"].max())

print("\nStatistiques :")
print(df["taux_de_ponctualite"].describe())

Dimensions finales : (1290, 6)

Valeurs manquantes :
date                   0
ligne                  0
taux_de_ponctualite    0
annee                  0
mois                   0
trimestre              0
dtype: int64

Doublons :
0

Doublons par date et ligne :
0

Période couverte :
2013-01-01 00:00:00
2026-10-01 00:00:00

Statistiques :
count    1290.000000
mean       91.299169
std         4.633926
min        44.925000
25%        88.975000
50%        92.118000
75%        94.679000
max        99.411000
Name: taux_de_ponctualite, dtype: float64


In [14]:
#export en un nouveau fichier CSV propre
df.to_csv(
    "../data/data_nettoyees.csv",
    index=False,
    encoding="utf-8"
)

print("dataset nottoye exporté avec succès")

dataset nottoye exporté avec succès
